In [20]:
import os, sqlite3, random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

BASE = r"C:\Users\Tarane\Desktop\hospital-analytics"
DATA = os.path.join(BASE, "data")
DB   = os.path.join(BASE, "hospital_analytics.db")
os.makedirs(DATA, exist_ok=True)

try:
    conn.close()
except:
    pass

try:
    os.remove(DB)
except:
    pass

np.random.seed(42)
random.seed(42)

DEPARTMENTS = {
    "D01": "Cardiology",   "D02": "Orthopedics",
    "D03": "Neurology",    "D04": "Oncology",
    "D05": "Emergency",    "D06": "Pediatrics",
    "D07": "Radiology",    "D08": "General Surgery",
}
DIAGNOSES = [
    "Hypertension","Diabetes Type 2","Pneumonia","Fracture",
    "Appendicitis","Stroke","Cardiac Arrest","Knee Replacement",
    "Tumor Removal","Chemotherapy Session","MRI Scan",
    "Hip Replacement","Pediatric Fever","Asthma Attack",
]
INSURANCE_TYPES = ["Public (AOK)","Public (TK)","Private (DKV)","Private (Allianz)","None"]
CLAIM_STATUSES  = ["Approved","Denied","Pending","Partially Approved"]
START_DATE      = datetime(2023, 1, 1)
END_DATE        = datetime(2024, 12, 31)

DEPT_BASE_REVENUE = {
    "D01": 8000,  "D02": 12000, "D03": 9000,  "D04": 15000,
    "D05": 3000,  "D06": 4000,  "D07": 2500,  "D08": 11000,
}
DEPT_WEIGHTS = {
    "D01": 15, "D02": 15, "D03": 12, "D04": 12,
    "D05": 20, "D06": 10, "D07": 10, "D08": 6,
}

def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

dept_pool = []
for did, w in DEPT_WEIGHTS.items():
    dept_pool.extend([did] * w)

# Departments
df_departments = pd.DataFrame([{
    "dept_id": did, "dept_name": dname,
    "cost_center": f"CC-{did}",
    "head_physician": f"Dr. {random.choice(['Müller','Schmidt','Weber','Fischer','Wagner'])}",
    "floor": random.randint(1, 6),
    "bed_count": random.randint(10, 60),
} for did, dname in DEPARTMENTS.items()])

# Patients
rows = []
for i in range(1, 2001):
    adm = random_date(START_DATE, END_DATE)
    los = random.randint(1, 30)
    rows.append({
        "patient_id": f"P{i:05d}",
        "age": random.randint(1, 90),
        "gender": random.choice(["M","F"]),
        "dept_id": random.choice(dept_pool),
        "diagnosis": random.choice(DIAGNOSES),
        "admission_date": adm.date(),
        "discharge_date": (adm + timedelta(days=los)).date(),
        "length_of_stay": los,
        "insurance_type": random.choice(INSURANCE_TYPES),
        "readmitted": random.choices([0,1], weights=[85,15])[0],
    })
df_patients = pd.DataFrame(rows)

# Billing
rows = []
for i, p in df_patients.iterrows():
    base  = DEPT_BASE_REVENUE.get(p["dept_id"], 6000)
    total = round(base * (0.5 + p["length_of_stay"] * 0.15) * random.uniform(0.8, 1.3), 2)
    covered = 0.0
    if p["insurance_type"] != "None":
        if "Private" in p["insurance_type"]:
            covered = round(total * random.uniform(0.80, 0.95), 2)
        else:
            covered = round(total * random.uniform(0.60, 0.80), 2)
    bill_date = pd.to_datetime(p["discharge_date"])
    rows.append({
        "bill_id": f"B{i:05d}",
        "patient_id": p["patient_id"],
        "dept_id": p["dept_id"],
        "bill_date": str(p["discharge_date"]),
        "billing_month": bill_date.strftime("%Y-%m"),
        "billing_year": bill_date.year,
        "total_charges": total,
        "insurance_covered": covered,
        "out_of_pocket": round(total - covered, 2),
        "claim_status": "N/A" if p["insurance_type"] == "None"
                        else random.choices(CLAIM_STATUSES, weights=[70,10,12,8])[0],
        "paid": random.choices([1,0], weights=[88,12])[0],
    })
df_billing = pd.DataFrame(rows)

#  revenue for each depts
dept_revenue = df_billing.groupby("dept_id")["total_charges"].sum()

# Dept Costs — 60% revenue
dept_cost_total = (dept_revenue * 0.60).reset_index()
dept_cost_total.columns = ["dept_id", "total_cost"]

# Dept Costs
rows = []
for dept_id in DEPARTMENTS:
    total_cost_2yr = dept_cost_total[dept_cost_total["dept_id"] == dept_id]["total_cost"].values[0]
    monthly = total_cost_2yr / 24
    for month in pd.date_range("2023-01", "2024-12", freq="MS"):
        tc = round(monthly * random.uniform(0.97, 1.03), 2)
        rows.append({
            "cost_id": f"{dept_id}-{month.strftime('%Y%m')}",
            "dept_id": dept_id,
            "month": str(month.date()),
            "cost_month": month.strftime("%Y-%m"),
            "staff_cost": round(tc * 0.55, 2),
            "equipment_cost": round(tc * 0.15, 2),
            "medication_cost": round(tc * 0.20, 2),
            "overhead_cost": round(tc * 0.10, 2),
            "total_cost": tc,
        })
df_dept_costs = pd.DataFrame(rows)

# Save CSVs
df_departments.to_csv(os.path.join(DATA, "departments.csv"), index=False)
df_patients.to_csv(   os.path.join(DATA, "patients.csv"),    index=False)
df_billing.to_csv(    os.path.join(DATA, "billing.csv"),     index=False)
df_dept_costs.to_csv( os.path.join(DATA, "dept_costs.csv"),  index=False)

# Load to DB
conn = sqlite3.connect(DB)
df_departments.to_sql("departments", conn, if_exists="replace", index=False)
df_dept_costs.to_sql( "dept_costs",  conn, if_exists="replace", index=False)
df_patients.to_sql(   "patients",    conn, if_exists="replace", index=False)
df_billing.to_sql(    "billing",     conn, if_exists="replace", index=False)
conn.commit()

# ── Power BI exports ──────────────────────────────────────────────

# 1. Monthly Revenue
pd.read_sql('''
    SELECT billing_month AS month,
           ROUND(SUM(total_charges),2)     AS total_revenue,
           ROUND(SUM(insurance_covered),2) AS insurance_revenue,
           ROUND(SUM(out_of_pocket),2)     AS patient_revenue
    FROM billing GROUP BY billing_month ORDER BY billing_month
''', conn).to_csv(os.path.join(DATA, "powerbi_monthly_revenue.csv"), index=False)

# 2. Dept Performance — بدون JOIN به dept_costs
revenue_by_dept = pd.read_sql('''
    SELECT d.dept_name,
           COUNT(DISTINCT p.patient_id)        AS patient_count,
           ROUND(SUM(b.total_charges), 2)      AS total_revenue,
           ROUND(SUM(b.insurance_covered), 2)  AS insurance_revenue,
           ROUND(SUM(b.out_of_pocket), 2)      AS patient_revenue
    FROM billing b
    JOIN patients p    ON b.patient_id = p.patient_id
    JOIN departments d ON b.dept_id    = d.dept_id
    GROUP BY d.dept_name ORDER BY total_revenue DESC
''', conn)

cost_by_dept = pd.read_sql('''
    SELECT d.dept_name,
           ROUND(SUM(dc.total_cost), 2) AS total_cost
    FROM dept_costs dc
    JOIN departments d ON dc.dept_id = d.dept_id
    GROUP BY d.dept_name
''', conn)

df_perf = revenue_by_dept.merge(cost_by_dept, on="dept_name")
df_perf["gross_profit"]      = (df_perf["total_revenue"] - df_perf["total_cost"]).round(2)
df_perf["profit_margin_pct"] = ((df_perf["gross_profit"] / df_perf["total_revenue"]) * 100).round(1)
df_perf.to_csv(os.path.join(DATA, "powerbi_dept_performance.csv"), index=False)

# 3. Claims
pd.read_sql('''
    SELECT claim_status, COUNT(*) AS count
    FROM billing WHERE claim_status != 'N/A'
    GROUP BY claim_status
''', conn).to_csv(os.path.join(DATA, "powerbi_claims.csv"), index=False)

# 4. Length of Stay
pd.read_sql('''
    SELECT d.dept_name, ROUND(AVG(p.length_of_stay),1) AS avg_los
    FROM patients p JOIN departments d ON p.dept_id = d.dept_id
    GROUP BY d.dept_name ORDER BY avg_los DESC
''', conn).to_csv(os.path.join(DATA, "powerbi_length_of_stay.csv"), index=False)

conn.close()

print(df_perf[["dept_name","total_revenue","total_cost","profit_margin_pct"]].to_string(index=False))


      dept_name  total_revenue  total_cost  profit_margin_pct
       Oncology    10960056.47  6598178.01               39.8
    Orthopedics    10410460.88  6240103.58               40.1
     Cardiology     7233226.14  4378373.02               39.5
      Neurology     6354075.76  3796845.01               40.2
      Emergency     3547516.20  2110511.39               40.5
General Surgery     3306122.95  1981299.16               40.1
     Pediatrics     2610909.98  1562215.51               40.2
      Radiology     1462489.35   874837.41               40.2
